## Feature Engineering and Data Selection for QM9 Dataset

In [ ]:
import copy

from src.core.fingerprints import Fingerprints
import pandas as pd

df = pd.read_csv('../data/qm9.csv')

smiles = df['smiles'].to_numpy()
smiles

In [ ]:
fp, f_names = Fingerprints().apply(smiles=smiles,
                                             names=['ecfp', ], **{'ecfp': {'radius': 2, 'size': 128, 'count': True}})
df_fingerprints = pd.DataFrame(fp, columns=f_names)
threshold = 0.85
df_fingerprints = df_fingerprints[[col for col in df_fingerprints.columns if df_fingerprints[col].value_counts(normalize=True).iloc[0] <= threshold]]
df_fingerprints['smiles'] = df['smiles'].values
df_fingerprints

In [ ]:
df_fingerprints = df_fingerprints.drop_duplicates(subset=[c for c in df_fingerprints.columns if c != 'smiles']).reset_index(drop=True)
df_fingerprints

In [ ]:
df_fingerprints.to_csv('../data/data_qm9_ecfp.csv', index=False)

In [ ]:
df_fingerprints['non_zero_count'] = df_fingerprints.drop(columns=['smiles']).sum(axis=1) + df_fingerprints.drop(columns=['smiles']).astype(bool).sum(axis=1)

# Sort the DataFrame by the non-zero count in descending order and select the top rows
# The 'mergesort' kind is used for stable sorting
top_rows = df_fingerprints.sort_values(by='non_zero_count', ascending=False, kind='mergesort')
top_rows['non_zero_count']

In [ ]:
df = df_fingerprints[df_fingerprints['non_zero_count'] >= 35].reset_index(drop=True)
threshold = 0.85
df = df[[col for col in df.columns if df[col].value_counts(normalize=True).iloc[0] <= threshold]]
df

In [ ]:
df = df.drop(columns=['non_zero_count'])
df.to_csv('../data/data_qm9_ecfp.csv', index=False)

## Data exploration

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math

df = pd.read_csv('../data/data_qm9_ecfp.csv')
df = df.sample(n=200, replace=False, random_state=42).reset_index(drop=True)
print(len(df))
df_features = df.drop(columns=['smiles'])
ncols = 3
nrows = math.ceil(len(df_features.columns) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
axes = axes.flatten()
for i, feature in enumerate(df_features.columns):
    sns.histplot(data=df, x=feature, kde=True, bins=20, ax=axes[i])
    axes[i].set_title(f'Distribution of {feature}', fontsize=12)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency', fontsize=10)
for j in range(len(df_features.columns), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.suptitle('Distribution of Individual Features', y=1.0, fontsize=18)
plt.show()

In [ ]:
selected_features = [10, 12, 16, 30, 33, 39, 81, 94, 123]
f_names = [f'ecfp_feature_{i}' for i in selected_features]

plt.figure(figsize=(len(selected_features) * 4, len(selected_features) * 4))
pair_plot = sns.pairplot(df[f_names], kind='kde')

# Add a title for the entire figure.
pair_plot.fig.suptitle('Pairwise Relationships of Selected Features', y=1.02, fontsize=16)

# Display the plot.
plt.show()

In [ ]:
from itertools import combinations

feature_pairs = list(combinations(f_names, 2))
ncols = 3
nrows = math.ceil(len(feature_pairs) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
axes = axes.flatten()
for i, (f1, f2) in enumerate(feature_pairs):
    product = df[f1] * df[f2]
    sns.histplot(product, kde=True, bins=20, ax=axes[i])
    axes[i].set_title(f'Product of {f1} & {f2}', fontsize=12)
    axes[i].set_xlabel('Product Value')
    axes[i].set_ylabel('Frequency')

# Hide any unused subplots
for j in range(len(feature_pairs), len(axes)):
    axes[j].set_visible(False)

# Adjust layout and add a main title
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.suptitle('Distribution of Pairwise Feature Products', fontsize=18)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import copy
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/data_qm9_ecfp.csv')
df = df.sample(n=200, replace=False, random_state=42).reset_index(drop=True)
scaler = StandardScaler()

def simple_linear_function3(df: pd.DataFrame) -> pd.Series:
    selected_features = [30, 123, 10]
    f_30 = df[f'ecfp_feature_{selected_features[0]}']
    f_123 = df[f'ecfp_feature_{selected_features[1]}']
    f_10 = df[f'ecfp_feature_{selected_features[2]}']

    target = 2.5 * f_30 + 0.5 * f_123 - 3.5 * f_10
    return target

def simple_linear_function6(df: pd.DataFrame) -> pd.Series:
    selected_features = [30, 123, 10, 16, 81, 33]
    f_30 = df[f'ecfp_feature_{selected_features[0]}']
    f_123 = df[f'ecfp_feature_{selected_features[1]}']
    f_10 = df[f'ecfp_feature_{selected_features[2]}']
    f_16 = df[f'ecfp_feature_{selected_features[3]}']
    f_81 = df[f'ecfp_feature_{selected_features[4]}']
    f_33 = df[f'ecfp_feature_{selected_features[5]}']
    target = 8.5 * f_30 + 10.5 * f_123 - 3.5 * f_10 + 3 * f_16 - 2.5 * f_81 + 5.5 * f_33 + 30
    return target

def simple_linear_function9(df: pd.DataFrame) -> pd.Series:
    selected_features = [10, 12, 16, 30, 33, 39, 81, 94, 123]
    f_10 = df[f'ecfp_feature_{selected_features[0]}']
    f_12 = df[f'ecfp_feature_{selected_features[1]}']
    f_16 = df[f'ecfp_feature_{selected_features[2]}']
    f_30 = df[f'ecfp_feature_{selected_features[3]}']
    f_33 = df[f'ecfp_feature_{selected_features[4]}']
    f_39 = df[f'ecfp_feature_{selected_features[5]}']
    f_81 = df[f'ecfp_feature_{selected_features[6]}']
    f_94 = df[f'ecfp_feature_{selected_features[7]}']
    f_123 = df[f'ecfp_feature_{selected_features[8]}']

    target = -2.5 * f_10 + 3 * f_12 + 5 * f_16 - 7 * f_30 + 4.5 * f_33 - 1.5 * f_39 + 2.5 * f_81 - 0.5 * f_94 + 1.5 * f_123 + 30
    return target

def piecewise_linear_function3(df: pd.DataFrame) -> pd.Series:
    selected_features = [30, 123, 10]
    f_30 = df[f'ecfp_feature_{selected_features[0]}']
    f_123 = df[f'ecfp_feature_{selected_features[1]}']
    f_10 = df[f'ecfp_feature_{selected_features[2]}']

    conditions = [
        f_10 == 0,
        f_10 == 1,
        f_10 >= 2
    ]

    choices = [
        1.5 * f_30 + 5.5 * f_123,
        3.5 * f_30 - 1.5 * f_123,
        -2.5 * f_30 + 4.5 * f_123
    ]

    target = np.select(conditions, choices, default=0)
    return pd.Series(target)

def piecewise_linear_function6(df: pd.DataFrame) -> pd.Series:
    selected_features = [30, 123, 10, 16, 81, 33]
    f_30 = df[f'ecfp_feature_{selected_features[0]}']
    f_123 = df[f'ecfp_feature_{selected_features[1]}']
    f_10 = df[f'ecfp_feature_{selected_features[2]}']
    f_16 = df[f'ecfp_feature_{selected_features[3]}']
    f_81 = df[f'ecfp_feature_{selected_features[4]}']
    f_33 = df[f'ecfp_feature_{selected_features[5]}']

    X_not_scaled = scaler.inverse_transform(df)
    df_not_scaled = pd.DataFrame(X_not_scaled, columns=df.columns)
    f_10 = df_not_scaled[f'ecfp_feature_{selected_features[2]}']
    f_10 = f_10.round(0)

    conditions = [
        f_10 == 0,
        f_10 == 1,
        f_10 >= 2
    ]

    choices = [
        10.5 * f_30 + 6.5 * f_123 - 1.5 * f_81 + 30,
        5 * f_30 + 13 * f_123 - 2.5 * f_16 + 30,
        -1.5 * f_30 + 3.5 * f_123 + 15.5 * f_33 + 30
    ]

    target = np.select(conditions, choices, default=10000)
    return pd.Series(target)

def piecewise_linear_function9(df: pd.DataFrame) -> pd.Series:
    selected_features = [10, 12, 16, 30, 33, 39, 81, 94, 123]
    f_10 = df[f'ecfp_feature_{selected_features[0]}']
    f_12 = df[f'ecfp_feature_{selected_features[1]}']
    f_16 = df[f'ecfp_feature_{selected_features[2]}']
    f_30 = df[f'ecfp_feature_{selected_features[3]}']
    f_33 = df[f'ecfp_feature_{selected_features[4]}']
    f_39 = df[f'ecfp_feature_{selected_features[5]}']
    f_81 = df[f'ecfp_feature_{selected_features[6]}']
    f_94 = df[f'ecfp_feature_{selected_features[7]}']
    f_123 = df[f'ecfp_feature_{selected_features[8]}']

    conditions = [
        f_10 == 0,
        f_10 == 1,
        f_10 >= 2
    ]

    choices = [
        15 * f_30 + 0.5 * f_123 + 2 * f_16 - 6.5 * f_39 + 1.5 * f_81,
        2.5 * f_30 - 15 * f_123 + 1.5 * f_12 + 3.5 * f_33,
        -2.5 * f_30 + 4.5 * f_123 - 1.5 * f_94 + 7 * f_33 + 2 * f_39
    ]

    target = np.select(conditions, choices, default=10000)
    return pd.Series(target)

def nonlinear_function3(df: pd.DataFrame) -> pd.Series:
    selected_features = [30, 123, 10]
    f_30 = df[f'ecfp_feature_{selected_features[0]}']
    f_123 = df[f'ecfp_feature_{selected_features[1]}']
    f_10 = df[f'ecfp_feature_{selected_features[2]}']

    component1 = - 4 * f_123
    component2 = 6 * f_30 * f_10
    target = component1 + component2
    return target

def nonlinear_function6(df: pd.DataFrame) -> pd.Series:
    selected_features = [30, 123, 10, 16, 81, 33]
    f_30 = df[f'ecfp_feature_{selected_features[0]}']
    f_123 = df[f'ecfp_feature_{selected_features[1]}']
    f_10 = df[f'ecfp_feature_{selected_features[2]}']
    f_16 = df[f'ecfp_feature_{selected_features[3]}']
    f_81 = df[f'ecfp_feature_{selected_features[4]}']
    f_33 = df[f'ecfp_feature_{selected_features[5]}']

    component1 = -9.5 * f_10 + 2.5 * f_81 ** 2 - 3.5 * f_16
    component2 = 7.5 * f_30 * f_123
    component3 = 1.5 * f_33 ** 2 + 30
    return component1 + component2 + component3

def nonlinear_function9(df: pd.DataFrame) -> pd.Series:
    selected_features = [10, 12, 16, 30, 33, 39, 81, 94, 123]
    f_10 = df[f'ecfp_feature_{selected_features[0]}']
    f_12 = df[f'ecfp_feature_{selected_features[1]}']
    f_16 = df[f'ecfp_feature_{selected_features[2]}']
    f_30 = df[f'ecfp_feature_{selected_features[3]}']
    f_33 = df[f'ecfp_feature_{selected_features[4]}']
    f_39 = df[f'ecfp_feature_{selected_features[5]}']
    f_81 = df[f'ecfp_feature_{selected_features[6]}']
    f_94 = df[f'ecfp_feature_{selected_features[7]}']
    f_123 = df[f'ecfp_feature_{selected_features[8]}']

    component1 = 8.5 * f_123 ** 3 - 2 * f_30 + 5 * f_16**2
    component2 = - 10 * f_94 * f_81
    component3 = 2.5 * f_33 * f_39
    component4 = 1.5 * f_10 ** 2 + 3
    component5 = - 1.5 * f_12 ** 4

    return component1 + component2 + component3 + component4 + component5

# Create the feature matrix X
X = df.drop('smiles', axis=1)
X_data = scaler.fit_transform(X)
X = pd.DataFrame(X_data, columns=X.columns)

functions = {
    'simple_linear6': simple_linear_function6,
    'piecewise_linear_6': piecewise_linear_function6,
    'nonlinear_6': nonlinear_function6,
}

for name, func in functions.items():
    y = func(X)

    print(f"\nDescriptive Statistics for {name}:")
    print(y.describe())

    df_current = copy.deepcopy(df)
    df_current['target'] = y
    df_current.to_csv(f"../data/synthetic_data/qm9_{name}.csv", index=False)

    plt.figure(figsize=(10, 6))
    sns.histplot(y, kde=True, bins=40)
    plt.title(f'Distribution of {name}', fontsize=16)
    plt.xlabel('Target Value', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.grid(axis='y', alpha=0.5)
    plt.show()
    print('=' * 50)
